# Stage 1: Canopy Height (CH) Model Training

**Author** : Muhammad Wahyu Ramadhan
**GitHub**  : github.com/mwahyur46
**LinkedIn**: linkedin.com/in/mwahyur

Trains a Random Forest regressor to predict mangrove canopy height (CH)
from the full predictor stack (S2 + indices + S1 + slope), using GEDI L2A
RH98 as the training target.

**Workflow steps:**
1. Load CH training samples exported from `01_preprocessing.ipynb`
2. Train-test split (70/30, `random_state=42`)
3. Hyperparameter tuning via RandomizedSearchCV (5-fold CV)
4. Evaluate on the held-out test set (R2, RMSE, MAE, Bias)
5. Diagnostic plots: 1:1 scatter, feature importance, Pearson correlation matrix
6. Save the trained model for use in Stage 2 (AGB) and Stage 3 (inference)

Random Forest is used as the baseline algorithm for direct comparison with
the GEE `smileRandomForest` results. XGBoost or other algorithms can be
introduced in a later iteration once this baseline is established.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Local
# import sys
# sys.path.append('../src')

# Google Colab
import sys
sys.path.append('/content/drive/MyDrive/GitHub/mangrove-canopy-height-agb-estimation-gee/src')

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

!pip install leafmap localtileserver
!apt-get install -y gdal-bin -q
import leafmap
import localtileserver

import model_utils as mu


## 1. Load Training Data


In [ ]:
DATA_PATH = '/content/drive/MyDrive/GitHub/mangrove-canopy-height-agb-estimation-gee/data/raw/ch_samples_full.csv'
TARGET_COL = 'rh98'

df = pd.read_csv(DATA_PATH)

print(f'Loaded {len(df)} samples from: {DATA_PATH}')
df.head()


In [ ]:
print(f'Total samples: {len(df)}')
print(df[TARGET_COL].describe())

In [ ]:
# Filter outlier rh98
df = df[df['rh98'] <= 40]  # realistic threshold for West Kalimantan mangrove
print(f'Samples after outlier filter: {len(df)}')
print(df[TARGET_COL].describe())

In [ ]:
# Predictor columns: full feature set (S2 + indices + S1 + slope)
FEATURE_COLS = [
    'B2', 'B3', 'B4', 'B5', 'B6', 'B7', 'B8', 'B8A', 'B11', 'B12',
    'NDVI', 'NDWI', 'MNDWI', 'NDMI', 'CMRI', 'MVI', 'NDRE', 'SAVI', 'EVI',
    'VV', 'VH', 'slope',
]

print(f'Number of features: {len(FEATURE_COLS)}')
print(f'Target column: {TARGET_COL}')

# Sanity check: confirm all expected columns are present
missing = [c for c in FEATURE_COLS + [TARGET_COL] if c not in df.columns]
if missing:
    print(f'[WARNING] Missing columns: {missing}')
else:
    print('All expected columns present.')


In [ ]:
# ============================================================
# 2. EXPLORATORY DATA ANALYSIS
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Distribution of target variable
axes[0].hist(df[TARGET_COL], bins=40, color='#2e7d32', edgecolor='white')
axes[0].set_xlabel('Canopy Height / RH98 (m)')
axes[0].set_ylabel('Count')
axes[0].set_title('Target Distribution')

# Boxplot for outlier detection
axes[1].boxplot(df[TARGET_COL], vert=True, patch_artist=True,
                boxprops=dict(facecolor='#a5d6a7'))
axes[1].set_ylabel('Canopy Height / RH98 (m)')
axes[1].set_title('Outlier Detection')

# Cumulative distribution
sorted_vals = np.sort(df[TARGET_COL])
cdf = np.arange(1, len(sorted_vals) + 1) / len(sorted_vals)
axes[2].plot(sorted_vals, cdf, color='#2e7d32')
axes[2].set_xlabel('Canopy Height / RH98 (m)')
axes[2].set_ylabel('Cumulative Probability')
axes[2].set_title('Cumulative Distribution')

fig.tight_layout()
fig.savefig(f'/content/drive/MyDrive/GitHub/mangrove-canopy-height-agb-estimation-gee/images/ch_eda_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print('-' * 30)
print(f'Total samples : {len(df)}')
print(f'Mean          : {df[TARGET_COL].mean():.2f} m')
print(f'Median        : {df[TARGET_COL].median():.2f} m')
print(f'Std Dev       : {df[TARGET_COL].std():.2f} m')
print(f'Min           : {df[TARGET_COL].min():.2f} m')
print(f'Max           : {df[TARGET_COL].max():.2f} m')
print(f'Skewness      : {df[TARGET_COL].skew():.3f}')

In [ ]:
with rasterio.open(STACK_PATH) as src:
    print('Total bands:', src.count)
    print('CRS:', src.crs)
    print('Bounds:', src.bounds)
    print('GDF bounds:', gdf_proj.total_bounds)

In [ ]:
with rasterio.open(STACK_PATH) as src:
    print(src.descriptions)

In [ ]:
import json
import geopandas as gpd
import pandas as pd
import rasterio
from shapely.geometry import shape
import matplotlib.patches as mpatches

# ============================================================
# 3. SPATIAL DISTRIBUTION OF TRAINING SAMPLES
# ============================================================

STACK_PATH = '/content/drive/MyDrive/GitHub/mangrove-canopy-height-agb-estimation-gee/data/raw/base_stack_west_kalimantan.tif'
IMAGES_DIR = '/content/drive/MyDrive/GitHub/mangrove-canopy-height-agb-estimation-gee/images'

# Parse .geo column to GeoDataFrame
geometry = df['.geo'].apply(lambda x: shape(json.loads(x)))
gdf = gpd.GeoDataFrame(df, geometry=geometry, crs='EPSG:4326')

# Band order: B2 B3 B4 B5 B6 B7 B8 B8A B11 B12 ...
# B4=3, B8A=8, B11=9 (1-based rasterio index)
with rasterio.open(STACK_PATH) as src:
    b8a = src.read(8)
    b11 = src.read(9)
    b4  = src.read(3)
    extent = [src.bounds.left, src.bounds.right, src.bounds.bottom, src.bounds.top]
    crs_raster = src.crs

def normalize(band):
    p2, p98 = np.percentile(band[band > 0], [2, 98])
    return np.clip((band - p2) / (p98 - p2), 0, 1)

rgb = np.dstack([normalize(b8a), normalize(b11), normalize(b4)])
gdf_proj = gdf.to_crs(crs_raster)

# Spatial grid sampling: divide AOI into 3x2 grid, pick 1 point per cell
# ensures zoom panels are distributed across the full AOI extent
bounds = gdf_proj.total_bounds  # [xmin, ymin, xmax, ymax]
cols = np.linspace(bounds[0], bounds[2], 4)  # 3 columns
rows = np.linspace(bounds[1], bounds[3], 3)  # 2 rows

np.random.seed(42)
zoom_centers = []
for x0, x1 in zip(cols[:-1], cols[1:]):
    for y0, y1 in zip(rows[:-1], rows[1:]):
        cell = gdf_proj.cx[x0:x1, y0:y1]
        if len(cell) > 0:
            zoom_centers.append(cell.sample(1))

zoom_points = gpd.GeoDataFrame(pd.concat(zoom_centers), crs=gdf_proj.crs)

ZOOM_BUFFER = 0.05  # degrees (~5 km at equator)
colors_zoom = ['cyan', 'magenta', 'yellow', 'lime', 'orange', 'white']

fig, axes = plt.subplots(3, 2, figsize=(16, 21))
axes = axes.flatten()

# ---- Panel 1: Full scene with zoom boxes ----
ax = axes[0]
ax.imshow(rgb, extent=extent, origin='upper', aspect='equal')
scatter = ax.scatter(
    gdf_proj.geometry.x, gdf_proj.geometry.y,
    c=gdf_proj['rh98'], cmap='YlOrRd',
    s=4, alpha=0.7, linewidths=0,
)
for i, (_, row) in enumerate(zoom_points.iterrows()):
    cx, cy = row.geometry.x, row.geometry.y
    rect = mpatches.Rectangle(
        (cx - ZOOM_BUFFER, cy - ZOOM_BUFFER),
        ZOOM_BUFFER * 2, ZOOM_BUFFER * 2,
        linewidth=1.5, edgecolor=colors_zoom[i], facecolor='none',
        label=f'Zoom {i + 1}'
    )
    ax.add_patch(rect)
cbar = fig.colorbar(scatter, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label('RH98 (m)')
ax.set_title('Full Scene\n(B8A = Red, B11 = Green, B4 = Blue)')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.legend(loc='lower right', fontsize=8)

# ---- Panels 2-6: Spatially distributed zoom areas ----
for i, (_, row) in enumerate(zoom_points.iterrows()):
    ax = axes[i + 1]
    cx, cy = row.geometry.x, row.geometry.y
    xmin, xmax = cx - ZOOM_BUFFER, cx + ZOOM_BUFFER
    ymin, ymax = cy - ZOOM_BUFFER, cy + ZOOM_BUFFER

    ax.imshow(rgb, extent=extent, origin='upper', aspect='equal')
    scatter_zoom = ax.scatter(
        gdf_proj.geometry.x, gdf_proj.geometry.y,
        c=gdf_proj['rh98'], cmap='YlOrRd',
        s=20, alpha=0.8, linewidths=0,
    )
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)

    # Border color matches zoom box in full scene panel
    for spine in ax.spines.values():
        spine.set_edgecolor(colors_zoom[i])
        spine.set_linewidth(2)

    cbar_z = fig.colorbar(scatter_zoom, ax=ax, fraction=0.03, pad=0.02)
    cbar_z.set_label('RH98 (m)')
    ax.set_title(f'Zoom {i + 1} (center: {cx:.2f}, {cy:.2f})')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')

fig.suptitle(
    'GEDI L2A Training Samples on False Color Composite (B8A, B11, B4)',
    fontsize=14, y=1.01
)
fig.tight_layout()
for ax in axes:
    ax.yaxis.set_tick_params(rotation=90)
fig.savefig(f'{IMAGES_DIR}/ch_sample_spatial_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

print('-' * 30)
print(f'Total samples plotted : {len(gdf)}')
print(f'Raster CRS            : {crs_raster}')
print(f'Zoom panels           : {len(zoom_points)}')

## 2. Train-Test Split (70/30)


In [ ]:
# Split train-test with 70:30
X = df[FEATURE_COLS].values
y = df[TARGET_COL].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=mu.RANDOM_STATE
)

print(f'Train shape: X={X_train.shape}, y={y_train.shape}')
print(f'Test shape : X={X_test.shape}, y={y_test.shape}')


## 3. Hyperparameter Tuning (RandomizedSearchCV)

Random Forest with a standard search space over `n_estimators`, `max_depth`,
`min_samples_split`, `min_samples_leaf`, `max_features`, and `bootstrap`.
5-fold cross-validation, scored on R2.


In [ ]:
import time

start_time = time.time()

search = mu.train_rf_random_search(
    X_train, y_train,
    param_dist=mu.RF_PARAM_DIST,
    n_iter=100,
    cv=5,
    n_jobs=-1,
    scoring='r2',
    random_state=mu.RANDOM_STATE,
    verbose = 2
)

ch_model = search.best_estimator_

end_time = time.time()
print(f'Execution time: {(end_time - start_time):.2f} seconds')


## 4. Test Set Evaluation


In [ ]:
y_pred = ch_model.predict(X_test)

ch_metrics = mu.evaluate_regression(y_test, y_pred, unit='m', label='Canopy Height (Stage 1)')


## 5. Diagnostic Plots

1:1 observed vs predicted scatter, feature importance, and Pearson
correlation matrix across predictors and target.


In [ ]:
ch_importance = mu.get_feature_importance(ch_model, FEATURE_COLS)
ch_importance


In [ ]:
fig = mu.plot_model_diagnostics(
    y_test, y_pred, ch_importance,
    unit='m', label='Canopy Height (Stage 1)', metrics=ch_metrics,
)
fig.savefig('/content/drive/MyDrive/GitHub/mangrove-canopy-height-agb-estimation-gee/images/ch_model_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
mu.plot_correlation_matrix(
    df, columns=FEATURE_COLS + [TARGET_COL],
    title='Pearson Correlation Matrix: CH Predictors and Target', ax=ax,
)
fig.savefig('/content/drive/MyDrive/GitHub/mangrove-canopy-height-agb-estimation-gee/images/ch_correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()


## 6. Save Model

Saved as `.pkl` via `joblib` for reuse in Stage 2 (AGB, as the `CH_m`
predictor source) and Stage 3 (wall-to-wall inference).


In [ ]:
MODEL_PATH = '/content/drive/MyDrive/GitHub/mangrove-canopy-height-agb-estimation-gee/models/ch_rf_model.pkl'

joblib.dump(ch_model, MODEL_PATH)

print(f'Model saved: {MODEL_PATH}')
print(f'Best params: {search.best_params_}')


## Summary

| Metric | Value |
|--------|------:|
| R2     | see output above |
| RMSE   | see output above |
| MAE    | see output above |
| Bias   | see output above |

Proceed to `03_agb_model_training.ipynb`. This notebook's saved model
(`ch_rf_model.pkl`) will be used to generate the `CH_m` predictor for the
AGB training data.
